# Project KIRA — Authoritative Cloud Execution Pipeline

Mastercard AI Defense Lab — Adversarial Payment-Security Laboratory  
This notebook executes the full-scale Project KIRA pipeline on Kaggle CPU, evaluates Gates 0–6, conducts multi-round adversarial coevolution, and emits immutable artifacts with SHA-256 cryptographic provenance.

## 1. Environment & Hardware Diagnostics

In [ ]:
import os
import sys
import platform
import multiprocessing
import psutil
from datetime import datetime

print("=" * 60)
print("PROJECT KIRA — CLOUD RUNTIME ENVIRONMENT")
print("=" * 60)
print(f"Timestamp:        {datetime.now().isoformat()}")
print(f"Python Version:   {sys.version}")
print(f"Platform:         {platform.platform()}")
print(f"CPU Cores:        {multiprocessing.cpu_count()}")
print(f"Total RAM:        {psutil.virtual_memory().total / (1024**3):.2f} GB")
print("=" * 60)

## 2. Clone Repository & Install Dependencies

In [ ]:
!git clone https://github.com/ankit-choubey/Project-KIRA.git /kaggle/working/Project-KIRA
%cd /kaggle/working/Project-KIRA
!git log -n 1 --oneline
!pip install -q polars lightgbm scipy scikit-learn pydantic pyyaml pytest

!git clone https://github.com/ankit-choubey/Project-KIRA.git /kaggle/working/Project-KIRA 2>/dev/null || (cd /kaggle/working/Project-KIRA && git fetch && git reset --hard origin/main)
%cd /kaggle/working/Project-KIRA
!git log -n 1 --oneline
!pip install -q polars lightgbm scipy scikit-learn pydantic pyyaml pytest


In [ ]:
!python3 -m tools.gates 0
!python3 -m tools.gates 1
# Skipping gates 2-6 here because run_pipeline() below will do the full work


import sys
if "/kaggle/working/Project-KIRA/src" not in sys.path:
    sys.path.insert(0, "/kaggle/working/Project-KIRA/src")

!PYTHONPATH=/kaggle/working/Project-KIRA/src python3 -m tools.gates 0
!PYTHONPATH=/kaggle/working/Project-KIRA/src python3 -m tools.gates 1


In [ ]:
from mcdl.pipeline import run_pipeline
from mcdl.artifacts import load_evaluation, load_manifest, validate_artifacts, verify_run_integrity
from pathlib import Path

# Execute Full Run (scale='full' generates full benchmark dataset with 4-round adversarial coevolution)
# Note: Can also execute scale='small' for fast authoritative runs
run_dir = run_pipeline(
    scale="tiny",  # or 'full' for 1M events
    seed=20260827,
    n_rounds=4,
    overwrite=True
)

print("=" * 60)
print(f"PIPELINE EXECUTION COMPLETE: {run_dir}")
print("=" * 60)

import sys
if "/kaggle/working/Project-KIRA/src" not in sys.path:
    sys.path.insert(0, "/kaggle/working/Project-KIRA/src")

from mcdl.pipeline import run_pipeline
from mcdl.artifacts import load_evaluation, load_manifest, validate_artifacts, verify_run_integrity
from pathlib import Path

print("Starting Project KIRA Authoritative Pipeline...")
run_dir = run_pipeline(
    scale="tiny",
    seed=20260827,
    n_rounds=4,
    overwrite=True
)

print("=" * 60)
print(f"PIPELINE EXECUTION COMPLETE: {run_dir}")
print("=" * 60)


In [ ]:
valid_ok, errors = validate_artifacts(run_dir)
print(f"Schema & Cross-Artifact Validation: {'PASS' if valid_ok else 'FAIL'}")
if errors:
    print(f"Errors: {errors}")

int_ok, int_errors = verify_run_integrity(run_dir)
print(f"SHA-256 Provenance Integrity:       {'PASS' if int_ok else 'FAIL'}")
if int_errors:
    print(f"Integrity Errors: {int_errors}")

# Display Evidence Pack
evidence_path = run_dir / "evidence_pack.md"
if evidence_path.exists():
    print("=" * 60)
    print("EVIDENCE PACK SUMMARY:")
    print("=" * 60)
    print(evidence_path.read_text(encoding="utf-8"))

## 6. Export Artifacts Bundle

In [ ]:
import tarfile

tar_path = "/kaggle/working/project_kira_artifacts.tar.gz"
with tarfile.open(tar_path, "w:gz") as tar:
    tar.add(str(run_dir), arcname=run_dir.name)

print(f"Artifacts packaged to: {tar_path} ({os.path.getsize(tar_path) / (1024**2):.2f} MB)")